# Bidirectional LSTM on IMDB
Diana Nadia Tamayo Celada

El análisis de sentimientos es un proceso que implica determinar la actitud o emoción expresada en un fragmento de texto. Este problema se puede plantear como un problema de clasificación, donde el objetivo es asignar una etiqueta a cada texto que indique su sentimiento predominante, como positivo, negativo o neutral. El proceso de clasificación comienza con el preprocesamiento de datos, que incluye la tokenización para dividir el texto en palabras o frases significativas, la normalización para convertir el texto a minúsculas y eliminar puntuación y palabras irrelevantes, y la vectorización para convertir el texto en una representación numérica, como bolsas de palabras o embeddings.

Una vez preprocesados los datos, se entrena un modelo de clasificación utilizando un conjunto de datos etiquetado para que el modelo pueda aprender a distinguir entre diferentes sentimientos. Posteriormente, se evalúa el modelo en un conjunto de datos de prueba para medir su precisión y capacidad de generalización. Finalmente, el modelo entrenado se aplica a nuevos textos para predecir su sentimiento.

El análisis de sentimientos es un problema de cómputo cognitivo porque involucra la simulación de procesos cognitivos humanos, como la comprensión y la interpretación del lenguaje natural. Requiere que las máquinas entiendan el contexto y las sutilezas del lenguaje humano, lo cual es un aspecto central de la cognición humana. Además, las máquinas deben interpretar las emociones y actitudes subyacentes en el texto, una tarea que los humanos realizan de manera intuitiva pero que es compleja para las máquinas. El lenguaje humano es inherentemente ambiguo y dependiente del contexto, por lo que los sistemas de cómputo cognitivo deben manejar estas ambigüedades para realizar un análisis preciso.

Los modelos de análisis de sentimientos deben aprender de los datos y adaptarse a nuevos patrones de lenguaje, similar a cómo los humanos aprenden y adaptan su comprensión del lenguaje. Además, el análisis de sentimientos mejora la interacción entre humanos y computadoras al permitir que las máquinas respondan de manera más adecuada a las emociones humanas, lo que es crucial en aplicaciones como asistentes virtuales y chatbots. En resumen, el análisis de sentimientos no solo es un problema de clasificación técnica, sino que también representa un desafío en la simulación de capacidades cognitivas humanas, lo que lo convierte en un área clave dentro del cómputo cognitivo.

## Setup

In [10]:
import numpy as np
import keras
from keras import layers
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy import stats
import matplotlib.pyplot as plt

max_features = 20000  # Only consider the top 20k words
maxlen = 200  # Only consider the first 200 words of each movie review

## Build the model

In [3]:
# Input for variable-length sequences of integers
inputs = keras.Input(shape=(None,), dtype="int32")
# Embed each integer in a 128-dimensional vector
x = layers.Embedding(max_features, 128)(inputs)
# Add 2 bidirectional LSTMs
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
# Add a classifier
outputs = layers.Dense(1, activation="sigmoid")(x)
model_lstm = keras.Model(inputs, outputs)
model_lstm.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,757,761 (10.52 MB)

 Trainable params: 2,757,761 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

## Load the IMDB movie review sentiment data

In [32]:
# First, load training and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=max_features
)

# Then, if you need a validation set, split it from the training data
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)
print(len(x_train), "Training sequences")
print(len(x_val), "Validation sequences")
print(len(y_test),"Test sequences")
# Use pad_sequence to standardize sequence length:
# this will truncate sequences longer than 200 words and zero-pad sequences shorter than 200 words.
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)
x_test=keras.utils.pad_sequences(x_test, maxlen=maxlen)

20000 Training sequences
5000 Validation sequences
25000 Test sequences


## Train and evaluate the model

You can use the trained model hosted on [Hugging Face Hub](https://huggingface.co/keras-io/bidirectional-lstm-imdb)
and try the demo on [Hugging Face Spaces](https://huggingface.co/spaces/keras-io/bidirectional_lstm_imdb).

In [21]:
# Input for variable-length sequences of integers
inputs = keras.Input(shape=(None,), dtype="int32")
# Embed each integer in a 128-dimensional vector
x = layers.Embedding(max_features, 128)(inputs)
# Add 2 bidirectional LSTMs
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
# Add a classifier
outputs = layers.Dense(1, activation="sigmoid")(x)
model_lstm = keras.Model(inputs, outputs)
model_lstm.summary()
model_lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_lstm.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_7 (Embedding)         │ (None, None, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,757,761 (10.52 MB)

 Trainable params: 2,757,761 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 37s 43ms/step - accuracy: 0.7479 - loss: 0.4869 - val_accuracy: 0.8507 - val_loss: 0.3470
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 34s 43ms/step - accuracy: 0.9210 - loss: 0.2084 - val_accuracy: 0.8567 - val_loss: 0.3405


In [23]:

def create_mlp_model(max_features, optimizer='adam'):
    inputs = keras.Input(shape=(None,), dtype="int32")
    x = layers.Embedding(max_features, 128)(inputs)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [28]:
def compare_mlp_with_lstm(
    x_train, y_train, x_val, y_val, model_lstm, max_features
):
    """
    Compare MLP models with different optimizers against an LSTM model.

    Args:
        x_train: Training data
        y_train: Training labels
        x_val: Validation data
        y_val: Validation labels
        x_test: Test data
        y_test: Test labels
        model_lstm: Pre-trained LSTM model
        max_features: Vocabulary size for embedding layer
    """
    optimizers = ['adam', 'sgd', 'rmsprop', 'adagrad']
    mlp_models = {}
    mlp_histories = {}
    test_results = {}

    for opt in optimizers:
        print(f"\nEntrenando MLP con optimizador {opt}:")
        mlp_models[opt] = create_mlp_model(max_features, optimizer=opt)
        history = mlp_models[opt].fit(
            x_train, y_train,
            batch_size=32,
            epochs=2,
            validation_data=(x_val, y_val),
            verbose=1
        )
        mlp_histories[opt] = history.history

        test_loss, test_acc = mlp_models[opt].evaluate(x_test, y_test, verbose=0)
        test_results[opt] = test_acc

    lstm_val_acc = model_lstm.history.history['val_accuracy'][-1]
    mlp_val_accs = [hist['val_accuracy'][-1] for hist in mlp_histories.values()]

    t_stat, p_value = stats.ttest_1samp(mlp_val_accs, lstm_val_acc)

    print("\nResultados de la Comparación Estadística:")
    print(f"Precisión de Validación LSTM: {lstm_val_acc:.4f}")
    print("\nPrecisión de Validación MLP:")
    for opt, hist in mlp_histories.items():
        print(f"{opt}: {hist['val_accuracy'][-1]:.4f}")

    print("\nPrecisión en Conjunto de Prueba MLP:")
    for opt, acc in test_results.items():
        print(f"{opt}: {acc:.4f}")

    print(f"\nEstadística-t: {t_stat:.4f}")
    print(f"Valor-p: {p_value:.4f}")

    plt.figure(figsize=(10, 6))
    for opt, hist in mlp_histories.items():
        plt.plot(hist['val_accuracy'], label=f'MLP-{opt}')
    plt.axhline(y=lstm_val_acc, color='r', linestyle='--', label='LSTM')
    plt.title('Comparación de Modelos: Precisión de Validación')
    plt.xlabel('Época')
    plt.ylabel('Precisión de Validación')
    plt.legend()
    plt.grid(True)
    plt.savefig('comparacion_modelos.png')
    plt.close()

    return mlp_models, mlp_histories, test_results

In [33]:
mlp_models, mlp_histories, test_results = compare_mlp_with_lstm(
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    model_lstm=model_lstm,
    max_features=max_features
)


Entrenando MLP con optimizador adam:
Epoch 1/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6590 - loss: 0.5839 - val_accuracy: 0.8420 - val_loss: 0.3402
Epoch 2/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9029 - loss: 0.2530 - val_accuracy: 0.8500 - val_loss: 0.3461

Entrenando MLP con optimizador sgd:
Epoch 1/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.5030 - loss: 0.6930 - val_accuracy: 0.5110 - val_loss: 0.6930
Epoch 2/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.5112 - loss: 0.6930 - val_accuracy: 0.4874 - val_loss: 0.6931

Entrenando MLP con optimizador rmsprop:
Epoch 1/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5172 - loss: 0.6908 - val_accuracy: 0.7200 - val_loss: 0.5710
Epoch 2/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7434 - loss: 0.5287 - val_accuracy: 0.8338 - val_loss: 0.3836

Entrenando MLP con optimizador adagrad:
Epoch 1/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5055 - loss: 0.6

## Análisis de Resultados
El modelo LSTM alcanzó una precisión de validación de 0.8567, lo que indica un buen rendimiento en el conjunto de validación. Entre los modelos MLP, el optimizador Adam logró la mejor precisión de validación con 0.8500, seguido de RMSprop con 0.8338. Ambos están relativamente cerca del rendimiento del LSTM.
Los optimizadores SGD y Adagrad mostraron un rendimiento significativamente inferior, con precisiones de 0.4874 y 0.5100 respectivamente, lo que sugiere que estos optimizadores no son adecuados para este tipo de arquitectura MLP en este problema específico.

Similar a los resultados de validación, Adam y RMSprop también mostraron un mejor rendimiento en el conjunto de prueba, con precisiones de 0.8334 y 0.8272 respectivamente.
Los resultados de SGD y Adagrad en el conjunto de prueba fueron bajos, reflejando su bajo rendimiento en la validación.

La prueba t de una muestra comparó las precisiones de validación de los modelos MLP con la del modelo LSTM. La estadística-t fue de -1.8782 y el valor-p fue de 0.1570.
Un valor-p de 0.1570 indica que no hay una diferencia estadísticamente significativa entre las precisiones de validación del LSTM y los MLP con un nivel de significancia común de 0.05. Esto sugiere que, aunque el LSTM tiene una ligera ventaja en precisión, la diferencia no es lo suficientemente grande como para ser considerada significativa en términos estadísticos.

## Conclusión
El modelo LSTM mostró un rendimiento ligeramente superior en comparación con los modelos MLP, especialmente cuando se utilizan optimizadores como Adam y RMSprop. Sin embargo, la diferencia no es estadísticamente significativa, lo que sugiere que los MLP con optimizadores adecuados pueden ser una alternativa viable dependiendo de otros factores como la complejidad del modelo y el tiempo de entrenamiento. Los optimizadores SGD y Adagrad no son recomendables para esta configuración de MLP en este problema específico debido a su bajo rendimiento.

La elección del optimizador puede tener un impacto significativo en el rendimiento del modelo. Optimizadores como Adam y RMSprop son generalmente más robustos y adaptativos, lo que los hace adecuados para una amplia gama de problemas. Por otro lado, optimizadores como SGD y Adagrad pueden requerir ajustes más cuidadosos y pueden no ser tan efectivos en problemas con alta variabilidad o características densas. En este caso, la adaptabilidad de Adam y RMSprop les permitió manejar mejor las características del conjunto de datos, resultando en un mejor rendimiento.